# **Import Libraries and Prepare Dataset**

In [3]:
import pandas as pd
import numpy as np
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
import hdbscan
import re
import random
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.utils.dataframe import dataframe_to_rows

import warnings
warnings.filterwarnings('ignore')
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

In [4]:
df_modeling = pd.read_csv('df_modeling_BERT.csv')
df_modeling['sentiment'] = df_modeling['sentiment'].map({'positive': 1, 'negative': 0})
df_modeling.head()

,cleaned_text,sentiment
0,saya suka materi yang sudah disiapkan oleh pihak kampus karena memudahkan mahasiswa saya juga menyukai program enrichment yang disediakan kampus saya sehingga mahasiswa dapat belajar di ruang lingkup yang lebih luas,1
1,bisa bertemu dengan teman teman baru dan mendapatkan koneksi serta mendapatkan pelajaran yang berguna bagi saya kedepan nya,1
2,saya suka dengan makanan yang ada di dalam kampus saya terutama bakmi efata selain itu disekitar kampus juga banyak makanan enak,1
3,fasilitas kampus alam sutera sangat bagus pelajaran lab diajarkan oleh asisten yang sangat mengerti materi,1
4,saya suka dengan pertemanan nya solid mau saling bantu satu sama lain bagi bagi kisi kisi pas ujian terus saling ngajarin,1


In [5]:
df_pos = df_modeling[df_modeling['sentiment'] == 1]
df_neg = df_modeling[df_modeling['sentiment'] == 0]

In [6]:
def clean_text(text, negation=True):
  if negation:
      for phrase in ['sangat tidak menyukai', 'tidak menyukai','sangat tidak suka', 'tidak suka', 'kurang suka', 'kurang menyukai', 'ga suka', 'gak suka', 'ga menyukai', 'gak menyukai']:
          text = text.replace(phrase, '')

  for phrase in ['suka', 'sangat suka', 'menyukai', 'sangat menyukai']:
      text = text.replace(phrase, '')

  text = re.sub(r'\s+', ' ', text).strip()
  return text

In [7]:
# from exclude_words import exclude_stopwords
from nlp_id.stopword import StopWord
from nlp_id.tokenizer import Tokenizer

stopword = StopWord()
tokenizer = Tokenizer()

# Menambahkan kata untuk stop words
stop_words = stopword.get_stopword()
custom_stopwords = ['nya', 'ya', 'nih']
stop_words.append(custom_stopwords)

def text_preprocessing(text):

  # # Tokenisasi menggunakan Tokenizer dari nlp_id
  tokens = tokenizer.tokenize(text)

  filtered_tokens = [word for word in tokens if word not in stop_words]

  # Menghapus spasi yang berlebih
  text = re.sub(r'\s+', ' ', text).strip()

  processed_text = " ".join(filtered_tokens)

  return processed_text

In [8]:
df_pos['cleaned_text'] = df_pos['cleaned_text'].apply(clean_text, negation=False)
df_neg['cleaned_text'] = df_neg['cleaned_text'].apply(clean_text, negation=True)
df_pos['cleaned_text'] = df_pos['cleaned_text'].apply(text_preprocessing)
df_neg['cleaned_text'] = df_neg['cleaned_text'].apply(text_preprocessing)

In [9]:
texts_pos = df_pos["cleaned_text"].astype(str).tolist()
texts_neg = df_neg["cleaned_text"].astype(str).tolist()

# **BERTopic**

### Create Model

In [10]:
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

topic_model_pos = BERTopic(embedding_model=embedding_model, verbose=True)
topic_model_neg = BERTopic(embedding_model=embedding_model, verbose=True)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Fit Model

In [11]:
topics_pos, probs_pos = topic_model_pos.fit_transform(texts_pos)

2026-02-13 13:22:16,888 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

2026-02-13 13:22:17,799 - BERTopic - Embedding - Completed ✓
2026-02-13 13:22:17,800 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-13 13:22:26,609 - BERTopic - Dimensionality - Completed ✓
2026-02-13 13:22:26,610 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-13 13:22:26,624 - BERTopic - Cluster - Completed ✓
2026-02-13 13:22:26,627 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-13 13:22:26,644 - BERTopic - Representation - Completed ✓


In [12]:
topics_neg, probs_neg = topic_model_neg.fit_transform(texts_neg)

2026-02-13 13:22:29,975 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

2026-02-13 13:22:30,303 - BERTopic - Embedding - Completed ✓
2026-02-13 13:22:30,304 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-13 13:22:30,769 - BERTopic - Dimensionality - Completed ✓
2026-02-13 13:22:30,770 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-13 13:22:30,788 - BERTopic - Cluster - Completed ✓
2026-02-13 13:22:30,790 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-13 13:22:30,803 - BERTopic - Representation - Completed ✓


### Inspect Topics

In [13]:
topic_model_pos.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,289,0_teman_kampus_dosen_materi,"[teman, kampus, dosen, materi, belajar, perkuliahan, lingkungan, kuliah, mahasiswa, hal]","[dosen dosen kampus friendly materi mudah dipelajari teman teman pelajaran perkuliahan, perkuliahan lingkungan kampus nyaman kondusif belajar teman teman suportif aktif kegiatan pengalaman kuliah menyenangkan, teman teman mendukung membantu perkuliahan teman teman belajar bareng memahami materi]"
1,1,12,1_makanan_kampus_kantin_fasilitas,"[makanan, kampus, kantin, fasilitas, teman, area, makan, peluang, enak, pilihan]","[makanan fasilitas wilayah kampus kampus terpenuhi, daerah kampus makanan explore makanan jajan teman teman terdekat seru gak nyangka circle real sma, makanan kampus bakmi efata disekitar kampus makanan enak]"


In [14]:
topic_model_neg.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,189,0_dosen_kampus_mahasiswa_kelas,"[dosen, kampus, mahasiswa, kelas, kuliah, materi, hal, fasilitas, tugas, perkuliahan]","[perkuliahan hal i jadwal perkuliahan padat tugas tugas menumpuk bersamaan hal sulit membagi optimal mata kuliah penyampaian materi menarik aplikatif proses belajar menyenangkan berusaha mengambil pelajaran pengalaman, hal i kuliah dosen dosen membawakan materi interaksi terbatas mahasiswa interaksi terbatas mahasiswa cepat bosan memperhatikan materi pembelajaran, fasilitas mahasiswa tugas kelas berukuran besar mengurangi interaksi dosen mahasiswa]"
1,1,64,1_toilet_fasilitas_tisu_kampus,"[toilet, fasilitas, tisu, kampus, wifi, bersih, tissue, nyaman, gedung, lambat]","[fasilitas toilet bersih tisu mencukupi wifi terkoneksi tugas lancar kampus, fasilitas toilet bersih tisu mencukupi wifi terkoneksi tugas lancar kampus, fasilitas toilet bersih tisu mencukupi wifi terkoneksi tugas lancar kampus]"
2,2,48,2_wifi_kampus_koneksi_susah,"[wifi, kampus, koneksi, susah, absensi, fasilitas, internet, jaringan, terkoneksi, lambat]","[fasilitas kampus wifi terputus koneksi jaringan lancar, wifi disediakan kampus koneksi jaringan kesulitan tugas absen, koneksi jaringan wifi kampus wifi terkonek aktivitas kampus kesulitan termaksa memakai data hotspot teman aktivitas krusial absensi kelas wifi]"


# **Model Evaluation** (Coherence and Diversity)

In [18]:
from itertools import combinations

import rbo
import itertools
import numpy as np

from gensim.utils import simple_preprocess
from gensim import corpora
from gensim.models.coherencemodel import CoherenceModel

In [19]:
def get_all_topics(topic_model_pos, topic_model_neg):
    topics_pos = topic_model_pos.get_topics()
    topics_neg = topic_model_neg.get_topics()

    all_topics_pos = [[word for word, _ in word_list] for word_list in topics_pos.values()]
    all_topics_neg = [[word for word, _ in word_list] for word_list in topics_neg.values()]

    print("="*5,"POSITIVE", "="*5)
    print(len(all_topics_pos))
    print(all_topics_pos)
    print("\n", "="*5,"NEGATIVE", "="*5)
    print(len(all_topics_neg))
    print(all_topics_neg)

    return all_topics_pos, all_topics_neg

all_topics_pos, all_topics_neg = get_all_topics(topic_model_pos, topic_model_neg)

===== POSITIVE =====
2
[['teman', 'kampus', 'dosen', 'materi', 'belajar', 'perkuliahan', 'lingkungan', 'kuliah', 'mahasiswa', 'hal'], ['makanan', 'kampus', 'kantin', 'fasilitas', 'teman', 'area', 'makan', 'peluang', 'enak', 'pilihan']]

 ===== NEGATIVE =====
3
[['dosen', 'kampus', 'mahasiswa', 'kelas', 'kuliah', 'materi', 'hal', 'fasilitas', 'tugas', 'perkuliahan'], ['toilet', 'fasilitas', 'tisu', 'kampus', 'wifi', 'bersih', 'tissue', 'nyaman', 'gedung', 'lambat'], ['wifi', 'kampus', 'koneksi', 'susah', 'absensi', 'fasilitas', 'internet', 'jaringan', 'terkoneksi', 'lambat']]


In [20]:
def calculate_coherence_score(texts, all_topics, coherence_scores):
  tokenized_docs = [simple_preprocess(doc) for doc in texts]

  # Build dictionary and corpus
  dictionary = corpora.Dictionary(tokenized_docs)
  corpus = [dictionary.doc2bow(text) for text in tokenized_docs]

  scores = {}

  # Compute all coherence types
  for coherence in coherence_scores:
      model = CoherenceModel(
          topics=all_topics,
          texts=tokenized_docs,
          corpus=corpus,
          dictionary=dictionary,
          coherence=coherence
      )
      scores[coherence] = round(model.get_coherence(), 3)

  for metric, score in scores.items():
    print(f"{metric}: {score}")
  return scores

def calculate_irbo(all_topics, topk=10):
    """
    all_topics: list of topic words (each topic is a list of top-k words)
    topk: how many top words to use per topic
    """
    T = len(all_topics)
    n = T * (T - 1) / 2
    rbo_sum = 0

    for i in range(1, T):
        for j in range(i):
            l1 = all_topics[i][:topk]
            l2 = all_topics[j][:topk]
            rbo_score = rbo.RankingSimilarity(l1, l2).rbo()
            rbo_sum += rbo_score

    irbo_score = 1 - (rbo_sum / n)
    return round(irbo_score, 3)

In [24]:
def evaluate_topics(texts, topic_model, coherence_types=['c_v'], topk=10):
    # Tokenized text for coherence
    tokenized_docs = [simple_preprocess(doc) for doc in texts]
    dictionary = corpora.Dictionary(tokenized_docs)
    corpus = [dictionary.doc2bow(text) for text in tokenized_docs]

    # Get all topic words
    all_topics = [ [word for word, _ in topic_model.get_topic(topic_id)[:topk]]
                   for topic_id in topic_model.get_topics().keys()
                   if topic_id != -1 and topic_model.get_topic(topic_id) is not None ]

    # Topic Coherence
    coherence_scores = {}
    for ctype in coherence_types:
        cm = CoherenceModel(topics=all_topics, texts=tokenized_docs, corpus=corpus, dictionary=dictionary, coherence=ctype)
        coherence_scores[ctype] = round(cm.get_coherence(), 3)

    # Topic Diversity (IRBO)
    irbo_score = calculate_irbo(all_topics, topk=topk)

    # Print all
    for ctype, score in coherence_scores.items():
        print(f"Coherence ({ctype}): {score}")
    print(f"IRBO Topic Diversity: {irbo_score}")

    return coherence_scores, irbo_score

In [26]:
print("positive sentiment topics model evaluation:")
evaluate_topics(texts_pos, topic_model_pos, coherence_types=['c_v'], topk=10)
print("negative sentiment topics model evaluation:")
evaluate_topics(texts_neg, topic_model_neg, coherence_types=['c_v'], topk=10)

positive sentiment topics model evaluation:
Coherence (c_v): 0.314
IRBO Topic Diversity: 0.723
negative sentiment topics model evaluation:
Coherence (c_v): 0.417
IRBO Topic Diversity: 0.787


({'c_v': np.float64(0.417)}, 0.787)